# S3 · Ejercicio 1 — Extraer coordenadas del padrón de domicilios (Sedona)

**Curso de Big Data · Dr. Abel Coronado** · Semana 3

> ⚙️ **Laboratorio BDP v7 (Codespaces o portable) · Spark 4.1 · Scala 2.13 · Sedona 1.9.1.**
> Un solo cuaderno para todas las plataformas, con el kernel **BDP v7 (Spark 4.1 · Python 3.12)**. Datos de práctica: recorte de Aguascalientes, cargado en HDFS con `python3 bdp_start.py datos`.

🎯 **Objetivo.** Leer el GeoParquet de domicilios desde HDFS, extraer lat/lon con Sedona (ST_X/ST_Y) y construir DIRECCION_COMPLETA; guardar el resultado en HDFS.

📦 **Datos.** `/data/raw/geocoder/integrado_ne.parquet` (recorte de Aguascalientes: 467,926 domicilios INEGI).

✅ **Prerrequisitos.** Datos cargados en HDFS con `python3 bdp_start.py datos`. Sedona 1.9.1 ya viene pre-cargado.

📝 **Entregable.** `/data/processed/geocoder_final_parquet` creado en HDFS (hdfs dfs -ls).

⬇️ **Repositorio:** https://github.com/abxda/bdp-lab-nube/tree/main/notebooks/semana_3

💬 **¿Un error?** Captura (celda + mensaje) y repórtalo por **Blackboard** (indica SO y paso).

---

> Lee los comentarios de cada celda: explican el porqué de cada paso.


In [ ]:
# BDP v7: ya viene instalado en el kernel 'BDP v7' -> !pip install pyarrow
# BDP v7: ya viene instalado en el kernel 'BDP v7' -> !pip install apache-sedona==1.8.0

In [ ]:
# Celda 1: Inicialización de Spark y Sedona (VERSIÓN CORREGIDA)

# 1. Importaciones necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr
from sedona.spark import SedonaContext

# 2. Detener cualquier sesión previa para una inicialización limpia
try:
    spark.stop()
    print("Sesión de Spark anterior detenida.")
except:
    pass

print("Configurando una nueva sesión de Spark con capacidades de Apache Sedona...")

# 3. Configuración y creación de la SparkSession con las versiones CORRECTAS
spark = (
    SparkSession.builder.appName("ExtraccionCoordenadasSedona")
    .master("local[*]")
    # Aumentamos la memoria disponible para el driver de Spark a 4GB
    # (Puedes ajustarlo a "2g" o "6g" según la RAM de tu equipo)
   .config("spark.driver.memory", "2g")  # BDP v7: recorte de un estado
   # BDP v7: Sedona 1.9.1 ya viene pre-cargado en el laboratorio (sin descargas de Maven)
   .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
   .config(
        "spark.kryo.registrator",
        "org.apache.sedona.core.serde.SedonaKryoRegistrator"
    )
   .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions")
   .config("spark.sql.ansi.enabled", "false").getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# 4. Inicialización del SedonaContext (ahora funcionará)
sedona = SedonaContext.create(spark)

print("✅ Sesión de Spark con Apache Sedona iniciada correctamente.")

In [ ]:
# Celda 2: Lectura de datos (CORREGIDA)

# Ruta al archivo que subimos a HDFS en la fase anterior
hdfs_path = "hdfs://localhost:9000/data/raw/geocoder/integrado_ne.parquet"

print(f"Leyendo archivo GeoParquet desde: {hdfs_path}")

# Leemos los datos. Sedona inferirá el esquema geométrico automáticamente.
df_domicilios = spark.read.format("geoparquet").load(hdfs_path)

# IMPORTANTE: Eliminamos la llamada a .cache()
# No es necesaria para este flujo de trabajo y es la causa del error de memoria.
# df_domicilios.cache()

# Realizamos una acción ligera como .printSchema() para verificar la carga inicial
print("\nEsquema cargado correctamente:")
df_domicilios.printSchema()

# Ahora, el .count() se ejecutará en la siguiente celda como parte de la transformación,
# pero ya sin el cuello de botella del .cache().
print(f"\nLectura inicial completada. La transformación comenzará ahora.")

In [ ]:
# Celda 3: Extracción de coordenadas

print("Extrayendo coordenadas usando las funciones nativas de Sedona ST_X y ST_Y...")

# Se asume que la columna de geometría se llama 'geometry'.
# ST_X extrae la Longitud.
# ST_Y extrae la Latitud.
df_con_coordenadas = df_domicilios.withColumn(
    "Longitud", expr("ST_X(geometry)")
).withColumn(
    "Latitud", expr("ST_Y(geometry)")
)

print("¡Extracción completada! Verificando el resultado...")

# Verificación del Esquema: confirma la presencia de las nuevas columnas.
print("\nEsquema del DataFrame resultante:")
df_con_coordenadas.printSchema()

# Inspección de Datos: muestra las primeras 5 filas para una revisión visual.
print("\nMuestra de los datos con las nuevas columnas:")
df_con_coordenadas.select("NOMVIAL", "geometry", "Longitud", "Latitud").show(5, truncate=50)

In [ ]:
# Celda 4: Construcción de dirección y guardado del Parquet final
from pyspark.sql import functions as sf

print("Construyendo la columna 'DIRECCION_COMPLETA' para el geocodificador...")

# Concatenamos los campos para crear la dirección completa y estandarizada.
df_enriquecido = df_con_coordenadas.withColumn("DIRECCION_COMPLETA", sf.concat_ws(
    " ",
    sf.col("TIPOVIAL"), sf.col("NOMVIAL"), sf.col("NUMEXT"),
    sf.lit(","), sf.col("TIPOASEN"), sf.col("NOMASEN"),
    sf.lit(", C.P. "), sf.col("CP"),
    sf.lit(", "), sf.col("NOM_LOC"),
    sf.lit(", "), sf.col("NOM_MUN"),
    sf.lit(", "), sf.col("NOM_ENT")
))

# Seleccionamos solo las columnas que irán a Elasticsearch y eliminamos la geometría original.
df_final_para_guardar = df_enriquecido.select(
    "DIRECCION_COMPLETA",
    "Latitud",
    "Longitud",
    "NOM_MUN",
    "NOM_ENT",
    "CP"
).drop("geometry") # La columna 'geometry' ya no es necesaria

# Ruta de salida en HDFS para el archivo procesado.
hdfs_path_processed = "hdfs://localhost:9000/data/processed/geocoder_final_parquet"

print(f"Guardando el DataFrame final en: {hdfs_path_processed}")

# Guardamos el resultado en formato Parquet, sobrescribiendo si ya existe.
df_final_para_guardar.write.mode("overwrite").parquet(hdfs_path_processed)

print("✅ ¡Proceso completado! El archivo Parquet enriquecido está listo en HDFS.")

# Liberamos la memoria caché
df_domicilios.unpersist()

In [ ]:
# Celda 5: Verificación del resultado en HDFS

# Ruta al archivo Parquet que procesamos y guardamos en la celda anterior
hdfs_path_processed = "hdfs://localhost:9000/data/processed/geocoder_final_parquet"

print(f"Leyendo el DataFrame final desde: {hdfs_path_processed}")

# Leemos el archivo parquet. Spark inferirá el esquema automáticamente.
df_verificacion = spark.read.parquet(hdfs_path_processed)

print("\nMostrando 5 registros del DataFrame final para inspección visual:")
# Usamos truncate=False para asegurarnos de ver la dirección completa sin cortes.
df_verificacion.show(5, truncate=False)

print("\nVerificación del esquema final. Nota que ya no existe la columna 'geometry':")
df_verificacion.printSchema()



In [ ]:
# Celda 6: Descarga de HDFS al directorio local 'notebooks'

import os
from hdfs import InsecureClient

# --- 1. Definición de Rutas ---
# Ruta de origen en HDFS (el directorio que queremos descargar)
hdfs_source_path = "/data/processed/geocoder_final_parquet"

# Ruta de destino local. Usamos os.getcwd() para obtener la ruta actual del notebook
# y así asegurar que el destino sea siempre la carpeta 'notebooks'.
local_dest_path = os.path.join(os.getcwd(), "geocoder_final_parquet")

print(f"Directorio de origen en HDFS: {hdfs_source_path}")
print(f"Directorio de destino local: {local_dest_path}")

# --- 2. Conexión al cliente HDFS ---
# Nos conectamos al puerto HTTP del NameNode (9870 por defecto en BDPv4)

client = InsecureClient('http://localhost:9870')

# --- 3. Ejecución de la Descarga ---
print("\nIniciando la descarga desde HDFS...")

# El comando download toma el origen en HDFS y el destino local.
# overwrite=True borrará el directorio local si ya existe, para evitar errores en re-ejecuciones.
try:
    import shutil; shutil.rmtree(local_dest_path, ignore_errors=True)  # BDP v7: re-ejecutar no duplica filas
    client.download(hdfs_source_path, local_dest_path, overwrite=True)
    print("✅ ¡Descarga completada exitosamente!")
except Exception as e:
    print(f"❌ Ocurrió un error durante la descarga: {e}")


# --- 4. Verificación Local ---
print(f"\nContenido del directorio local '{local_dest_path}':")

# Verificamos que el directorio y sus archivos ahora existen localmente.
if os.path.exists(local_dest_path):
    # Listamos los archivos dentro del directorio descargado
    downloaded_files = os.listdir(local_dest_path)
    for file_name in downloaded_files:
        print(f"  - {file_name}")
else:
    print("El directorio no fue descargado.")

In [ ]:
# Celda 7: Consolidación a un único archivo Parquet con Pandas (OPCIONAL)

import pandas as pd
import os

# --- 1. Definir las rutas de origen y destino ---
# Ruta al directorio que contiene los múltiples archivos Parquet de Spark
source_directory = os.path.join(os.getcwd(), "geocoder_final_parquet")

# Nombre del nuevo archivo Parquet único que vamos a crear
output_single_file = os.path.join(os.getcwd(), "geocoder_mexico_completo.parquet")

print(f"Directorio de origen: {source_directory}")
print(f"Archivo de salida único: {output_single_file}")


# --- 2. Leer el directorio Parquet con Pandas ---
# Pandas es lo suficientemente inteligente como para leer todos los archivos 'part-*.parquet'
# dentro de un directorio si se le pasa la ruta del directorio.
print("\nLeyendo directorio Parquet con Pandas...")
try:
    # Usamos el motor 'pyarrow', que es rápido y el estándar de la industria.
    df_pandas = pd.read_parquet(source_directory, engine='pyarrow')
    print("Lectura completada exitosamente.")

    # --- 3. Verificar los datos cargados en Pandas ---
    print(f"\nVerificación del DataFrame de Pandas:")
    print(f"  - Forma del DataFrame (filas, columnas): {df_pandas.shape}")
    print(f"  - Uso de memoria: {df_pandas.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    print("\nMostrando las primeras 5 filas desde Pandas:")
    display(df_pandas.head())

    # --- 4. Guardar como un único archivo Parquet ---
    print(f"\nGuardando el DataFrame en un único archivo: '{output_single_file}'...")

    # Al guardar, es una buena práctica usar index=False para no guardar el índice de Pandas en el archivo.
    df_pandas.to_parquet(output_single_file, engine='pyarrow', index=False)

    print("\n✅ ¡Proceso completado! Se ha creado el archivo Parquet único.")

except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el directorio de origen en '{source_directory}'.")
    print("Asegúrate de haber ejecutado la celda de descarga de HDFS primero.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")